# 🚀 SmartStudyInstructor — Kaggle Wav2Lip Server

**Run Cell 1 FIRST** (setup), then **Cell 2** (server).

⚠️ Make sure you have **GPU enabled** in Kaggle settings.

In [ ]:
# ============================================================
# CELL 1 — SETUP (Run this FIRST, wait for it to finish)
# ============================================================
import os, shutil

print("🔧 Step 1/5: Installing system tools...")
!pip install -q pip==25.0.1 setuptools==69.5.1 wheel==0.45.1 --upgrade
!apt-get install -qq -y ffmpeg 2>/dev/null

print("\n📦 Step 2/5: Cloning Wav2Lip...")
os.chdir("/kaggle/working")

# If Wav2Lip exists but inference.py is missing, it's corrupted. Delete it.
if os.path.exists("Wav2Lip") and not os.path.exists("Wav2Lip/inference.py"):
    print("  🗑️ Found corrupted Wav2Lip folder. Deleting and re-cloning...")
    shutil.rmtree("Wav2Lip")

if not os.path.exists("Wav2Lip"):
    !git clone https://github.com/Rudrabha/Wav2Lip.git
    print("  ✅ Wav2Lip cloned")
else:
    print("  ✅ Wav2Lip already exists")

# Verify the critical file exists
assert os.path.exists("Wav2Lip/inference.py"), "❌ inference.py NOT FOUND!"
print("  ✅ inference.py verified")

# Create temp directory that Wav2Lip needs
os.makedirs("Wav2Lip/temp", exist_ok=True)
os.makedirs("Wav2Lip/checkpoints", exist_ok=True)

print("\n📥 Step 3/5: Downloading model weights...")
CKPT = "Wav2Lip/checkpoints/wav2lip_gan.pth"
if not os.path.exists(CKPT) or os.path.getsize(CKPT) < 100_000_000:
    if os.path.exists(CKPT):
        os.remove(CKPT)
    !wget -q 'https://huggingface.co/camenduru/Wav2Lip/resolve/main/checkpoints/wav2lip_gan.pth' -O {CKPT}

if os.path.exists(CKPT) and os.path.getsize(CKPT) > 100_000_000:
    print(f"  ✅ Checkpoint OK: {os.path.getsize(CKPT)/1e6:.1f} MB")
else:
    print("  ❌ Checkpoint download FAILED!")

# s3fd face detection weights
SFD = "Wav2Lip/face_detection/detection/sfd/s3fd.pth"
if not os.path.exists(SFD) or os.path.getsize(SFD) < 1_000_000:
    os.makedirs(os.path.dirname(SFD), exist_ok=True)
    !wget -q 'https://www.adrianbulat.com/downloads/python-fan/s3fd-619a316812.pth' -O {SFD}
    print(f"  ✅ s3fd weights OK")

print("\n⚙️ Step 4/5: Installing Python packages...")
!pip install -q librosa==0.9.1 numba==0.59.1 llvmlite==0.42.0 --prefer-binary
!pip install -q opencv-python-headless==4.8.1.78 torch torchvision torchaudio --prefer-binary
!pip install -q flask pyngrok requests pillow tqdm

print("\n🔧 Step 5/5: Patching Wav2Lip for compatibility...")

# Patch inference.py for PyTorch 2.x
inf_path = "Wav2Lip/inference.py"
with open(inf_path, "r") as f:
    code = f.read()
patched = False
if "torch.load(checkpoint_path)" in code:
    code = code.replace(
        "checkpoint = torch.load(checkpoint_path)",
        "checkpoint = torch.load(checkpoint_path, map_location='cpu', weights_only=False)"
    )
    patched = True
if 'torch.load(checkpoint_path, map_location="cuda")' in code:
    code = code.replace(
        'checkpoint = torch.load(checkpoint_path, map_location="cuda")',
        "checkpoint = torch.load(checkpoint_path, map_location='cpu', weights_only=False)"
    )
    patched = True
with open(inf_path, "w") as f:
    f.write(code)
if patched:
    print("  ✅ Patched torch.load for PyTorch 2.x")

# Patch audio.py for newer librosa
audio_path = "Wav2Lip/audio.py"
if os.path.exists(audio_path):
    with open(audio_path, "r") as f:
        acode = f.read()
    if "mel(sr, n_fft" in acode:
        acode = acode.replace("mel(sr, n_fft", "mel(sr=sr, n_fft=n_fft")
        with open(audio_path, "w") as f:
            f.write(acode)
        print("  ✅ Patched audio.py for librosa compatibility")

# Quick checkpoint verify
import torch
try:
    ckpt = torch.load(CKPT, map_location='cpu', weights_only=False)
    print(f"  ✅ Checkpoint loads correctly")
    del ckpt
except Exception as e:
    print(f"  ❌ Checkpoint load FAILED: {e}")

print("\n" + "="*50)
print("✅ SETUP COMPLETE — Now run Cell 2")
print("="*50)

In [ ]:
# ============================================================
# CELL 2 — LIPSYNC SERVER (Run AFTER Cell 1 finishes)
# ============================================================
import os, sys, base64, subprocess, traceback, threading
from flask import Flask, request, jsonify
from PIL import Image

# ╔══════════════════════════════════════════════════════════╗
# ║  👉 PASTE YOUR NGROK TOKEN BELOW                        ║
# ╚══════════════════════════════════════════════════════════╝
NGROK_TOKEN = "paste_your_ngrok_token_here"

# ── Configuration ──
JOBS_DIR = "/kaggle/working/lipsync_jobs"
WAV2LIP_DIR = "/kaggle/working/Wav2Lip"
CHECKPOINT = os.path.join(WAV2LIP_DIR, "checkpoints", "wav2lip_gan.pth")

os.makedirs(JOBS_DIR, exist_ok=True)
os.makedirs(os.path.join(WAV2LIP_DIR, "temp"), exist_ok=True)

# Verify setup
assert os.path.exists(os.path.join(WAV2LIP_DIR, "inference.py")), \
    "❌ Wav2Lip/inference.py NOT FOUND! Run Cell 1 first!"
assert os.path.exists(CHECKPOINT), \
    "❌ Checkpoint NOT FOUND! Run Cell 1 first!"
print("✅ Wav2Lip verified")

# Only 1 scene at a time (prevents GPU race conditions)
inference_lock = threading.Lock()

app = Flask(__name__)

def prepare_avatar(src, dst, size=720):
    """Resize avatar to square for Wav2Lip face detection."""
    img = Image.open(src).convert("RGB")
    w, h = img.size
    scale = min(size/w, size/h)
    nw, nh = int(w*scale), int(h*scale)
    img = img.resize((nw, nh), Image.LANCZOS)
    canvas = Image.new("RGB", (size, size), (0,0,0))
    canvas.paste(img, ((size-nw)//2, (size-nh)//2))
    canvas.save(dst, "PNG")
    print(f"  🖼️ Avatar: {w}x{h} → {size}x{size}")

def run_wav2lip(scene_id, audio_path, avatar_path, output_mp4):
    """Run Wav2Lip inference with proper paths."""
    # Clean old temp files
    temp_dir = os.path.join(WAV2LIP_DIR, "temp")
    for f in os.listdir(temp_dir):
        os.remove(os.path.join(temp_dir, f))
    
    # Prepare avatar
    prep = os.path.join(JOBS_DIR, f"{scene_id}_prep.png")
    prepare_avatar(avatar_path, prep)
    
    # Run inference FROM the Wav2Lip directory
    cmd = [
        sys.executable, "inference.py",
        "--checkpoint_path", os.path.join("checkpoints", "wav2lip_gan.pth"),
        "--face", prep,
        "--audio", audio_path,
        "--outfile", output_mp4,
        "--pads", "0", "10", "0", "0",
        "--resize_factor", "1",
        "--nosmooth",
    ]
    
    print(f"  🚀 Running Wav2Lip...")
    result = subprocess.run(
        cmd, capture_output=True, text=True,
        cwd=WAV2LIP_DIR,  # CRITICAL: run from Wav2Lip dir
        timeout=300
    )
    
    if result.stdout.strip():
        lines = result.stdout.strip().split('\n')[-5:]
        print(f"  📤 STDOUT: {'  '.join(lines)}")
    if result.stderr.strip():
        err = [l for l in result.stderr.split('\n') if l.strip() and 'libav' not in l and 'built with' not in l and 'configuration' not in l]
        if err:
            print(f"  ⚠️ STDERR: {'  '.join(err[-5:])}")
    print(f"  Return code: {result.returncode}")
    
    # Check output
    if os.path.exists(output_mp4) and os.path.getsize(output_mp4) > 1000:
        print(f"  ✅ Video: {os.path.getsize(output_mp4)} bytes")
        return True
    
    # Fallback: check temp/result.avi
    result_avi = os.path.join(temp_dir, "result.avi")
    if os.path.exists(result_avi):
        print(f"  Converting result.avi → mp4...")
        subprocess.run([
            "ffmpeg", "-y", "-i", result_avi,
            "-c:v", "libx264", "-preset", "ultrafast",
            "-pix_fmt", "yuv420p", output_mp4
        ], capture_output=True, timeout=60)
        if os.path.exists(output_mp4) and os.path.getsize(output_mp4) > 1000:
            return True
    
    print(f"  ❌ No output produced")
    print(f"  temp/ contents: {os.listdir(temp_dir)}")
    return False

@app.route("/health")
def health():
    import torch
    gpu = torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU"
    return jsonify({"status": "online", "gpu_name": gpu})

@app.route("/clear_cache")
def clear_cache():
    import torch, gc
    gc.collect()
    if torch.cuda.is_available(): torch.cuda.empty_cache()
    return jsonify({"status": "cleared"})

@app.route("/generate_lipsync", methods=["POST"])
def generate_lipsync():
    try:
        data = request.json
        scene_id = data.get("scene_id", "scene_01")
        
        # Save audio
        audio_path = os.path.join(JOBS_DIR, f"{scene_id}.wav")
        with open(audio_path, "wb") as f:
            f.write(base64.b64decode(data["audio_base64"]))
        print(f"  📁 Audio: {os.path.getsize(audio_path)} bytes")
        
        # Save avatar
        avatar_path = os.path.join(JOBS_DIR, f"{scene_id}_avatar.png")
        with open(avatar_path, "wb") as f:
            f.write(base64.b64decode(data["avatar_image_base64"]))
        print(f"  📁 Avatar: {os.path.getsize(avatar_path)} bytes")
        
        output_mp4 = os.path.join(JOBS_DIR, f"{scene_id}_sync.mp4")
        
        # Serialize GPU access
        print(f"🎬 {scene_id}: waiting for GPU...")
        with inference_lock:
            print(f"  🔓 GPU acquired")
            ok = run_wav2lip(scene_id, audio_path, avatar_path, output_mp4)
        
        if not ok:
            raise Exception(f"Wav2Lip produced no output for {scene_id}")
        
        with open(output_mp4, "rb") as f:
            video_b64 = base64.b64encode(f.read()).decode()
        
        print(f"  ✅ {scene_id} complete!")
        return jsonify({"status": "success", "lipsync_video_base64": video_b64})
        
    except Exception as e:
        print(f"🔥 Error: {traceback.format_exc()}")
        return jsonify({"status": "error", "error_message": str(e)}), 500

# ── Start Server ──
from pyngrok import ngrok

try:
    from kaggle_secrets import UserSecretsClient
    token = UserSecretsClient().get_secret("NGROK_AUTH_TOKEN") or NGROK_TOKEN
except:
    token = NGROK_TOKEN

if not token or "paste_your" in token:
    print("❌ Paste your ngrok token in NGROK_TOKEN variable!")
else:
    ngrok.set_auth_token(token)
    try:
        for t in ngrok.get_tunnels(): ngrok.disconnect(t.public_url)
    except: pass
    tunnel = ngrok.connect(5000)
    url = tunnel.public_url
    # Handle both string and NgrokTunnel object
    if hasattr(tunnel, 'public_url'):
        url = tunnel.public_url
    else:
        url = str(tunnel)
    # Extract clean URL if it contains extra text
    if 'https://' in url:
        import re
        match = re.search(r'(https://[^\s"]+\.ngrok[^\s"]*)', url)
        if match:
            url = match.group(1)
    print(f"\n🚀 PUBLIC URL: {url}")
    print(f"\n📋 Paste in backend/.env:")
    print(f"   CLOUD_RENDER_URL={url}")
    app.run(host="0.0.0.0", port=5000)